In [1]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

NOME_CSV = "NFL Play by Play 2009-2017 (v4).csv"
NOME_PARQUET = "NFL_Play_by_Play_2009-2017_tratado.parquet"

In [2]:
def localizar_csv():
    """Localiza o CSV no Kaggle, no diretório atual ou via kagglehub."""
    # 1) Caminhos usuais
    candidatos = [
        Path("/kaggle/input") / NOME_CSV,
        Path.cwd() / NOME_CSV,
        Path("/mnt/data") / NOME_CSV,
    ]
    for caminho in candidatos:
        if caminho.is_file():
            return caminho

    # 2) Busca recursiva em pastas comuns
    for base in [Path("/kaggle/input"), Path.cwd(), Path("/mnt/data")]:
        if base.exists():
            encontrados = list(base.rglob(NOME_CSV))
            if encontrados:
                return encontrados[0]

    # 3) Download via kagglehub (como no gabarito)
    try:
        import kagglehub
        raiz = Path(kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016"))
        encontrados = list(raiz.rglob(NOME_CSV))
        if encontrados:
            return encontrados[0]
    except Exception as erro:
        warnings.warn(f"kagglehub indisponível: {erro}")

    raise FileNotFoundError(
        f"Não encontrei '{NOME_CSV}'. Adicione o dataset ou ajuste o caminho."
    )

csv_path = localizar_csv()
print("Arquivo de origem:", csv_path)
print("Tamanho do CSV (MiB):", round(csv_path.stat().st_size / 1024**2, 2))

df = pd.read_csv(csv_path, low_memory=False)
print("Dimensões:", df.shape)

c:\Users\54176749822\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Arquivo de origem: C:\Users\54176749822\.cache\kagglehub\datasets\maxhorowitz\nflplaybyplay2009to2016\versions\6\NFL Play by Play 2009-2017 (v4).csv
Tamanho do CSV (MiB): 262.77
Dimensões: (407688, 102)


In [5]:
banco_inicial = {
    "linhas": df.shape[0],
    "colunas": df.shape[1],
    "csv_mb": csv_path.stat().st_size / 1024**2,
    "memoria_mb": df.memory_usage(deep=True).sum() / 1024**2,
    "ausentes": int(df.isna().sum().sum()),
    "duplicadas": int(df.duplicated().sum()),
}

pd.Series(banco_inicial, name="banco_inicial").round(2).to_frame()

,banco_inicial
linhas,407688.00
colunas,102.00
csv_mb,262.77
memoria_mb,380.54
ausentes,11505187.00
duplicadas,0.00


In [26]:
df.info(verbose=True)
df.info(memory_usage="deep", verbose=False)

<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Data columns (total 102 columns):
 #    Column                       Dtype  
---   ------                       -----  
 0    Date                         str    
 1    GameID                       int64  
 2    Drive                        int64  
 3    qtr                          int64  
 4    down                         float64
 5    time                         str    
 6    TimeUnder                    int64  
 7    TimeSecs                     float64
 8    PlayTimeDiff                 float64
 9    SideofField                  str    
 10   yrdln                        float64
 11   yrdline100                   float64
 12   ydstogo                      int64  
 13   ydsnet                       int64  
 14   GoalToGo                     float64
 15   FirstDown                    float64
 16   posteam                      str    
 17   DefensiveTeam                str    
 18   desc                         str 

In [22]:
df.head(3)

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,yrdln,yrdline100,ydstogo,ydsnet,GoalToGo,FirstDown,posteam,DefensiveTeam,desc,PlayAttempted,Yards.Gained,sp,Touchdown,ExPointResult,TwoPointConv,...,AwayTimeouts_Remaining_Pre,HomeTimeouts_Remaining_Post,AwayTimeouts_Remaining_Post,No_Score_Prob,Opp_Field_Goal_Prob,Opp_Safety_Prob,Opp_Touchdown_Prob,Field_Goal_Prob,Safety_Prob,Touchdown_Prob,ExPoint_Prob,TwoPoint_Prob,ExpPts,EPA,airEPA,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,30.0,30.0,0,0,0.0,NaN,PIT,TEN,R.Bironas kicks 67 yards from TEN 30 to PIT 3....,1,39,0,0,NaN,NaN,...,3,3,3,0.001506,0.179749,0.006639,0.281138,0.213700,0.003592,0.313676,0.0,0.0,0.323526,2.014474,NaN,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,42.0,58.0,10,5,0.0,0.0,PIT,TEN,(14:53) B.Roethlisberger pass short left to H....,1,5,0,0,NaN,NaN,...,3,3,3,0.000969,0.108505,0.001061,0.169117,0.293700,0.003638,0.423011,0.0,0.0,2.338000,0.077907,-1.068169,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,47.0,53.0,5,2,0.0,0.0,PIT,TEN,(14:16) W.Parker right end to PIT 44 for -3 ya...,1,-3,0,0,NaN,NaN,...,3,3,3,0.001057,0.105106,0.000981,0.162747,0.304805,0.003826,0.421478,0.0,0.0,2.415907,-1.402760,NaN,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009


In [7]:
print("Dimensões (linhas, colunas):", df.shape)
df.info(memory_usage="deep", verbose=False)

Dimensões (linhas, colunas): (407688, 102)
<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Columns: 102 entries, Date to Season
dtypes: float64(33), int64(31), str(38)
memory usage: 380.5 MB


In [10]:
print("Quantidade de colunas por tipo de dado:")
df.dtypes.map(lambda t:t.name).value_counts()

Quantidade de colunas por tipo de dado:


str        38
float64    33
int64      31
Name: count, dtype: int64

In [20]:
perfil = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "n_unicos": df.nunique(),
    "ausentes": df.isna().sum(),
    "pct_ausentes": (df.isna().mean() * 100).round(2),
    "mem_MiB": (df.memory_usage(deep=True, index=False) / (1024 * 1024)).round(2),
})
display(perfil.sort_values("mem_MiB", ascending=False).head(10))

,tipo,n_unicos,ausentes,pct_ausentes,mem_MiB
desc,str,375962,2,0.00,36.11
Date,str,426,0,0.00,7.00
time,str,901,224,0.05,5.10
Tackler1,str,3636,166134,40.75,5.09
PlayType,str,15,0,0.00,4.95
PassOutcome,str,2,239506,58.75,4.89
Passer_ID,str,334,237445,58.24,4.78
Receiver_ID,str,1488,250328,61.40,4.66
Passer,str,352,240045,58.88,4.53
Receiver,str,1386,246127,60.37,4.45


In [27]:
valores_ausentes = df.isna().sum().sort_values(ascending=False)
ausentes_pct = (df.isna().mean() * 100).sort_values(ascending=False)

tabela_ausentes = pd.DataFrame({
    "ausentes": valores_ausentes,
    "pct_ausentes": ausentes_pct.round(2),
})

tabela_ausentes = tabela_ausentes[tabela_ausentes["ausentes"] > 0]

print(f"Colunas com algum valor ausente: {len(tabela_ausentes)} de {df.shape[1]}")
tabela_ausentes.head(20)

Colunas com algum valor ausente: 65 de 102


,ausentes,pct_ausentes
DefTwoPoint,407664,99.99
BlockingPlayer,407571,99.97
TwoPointConv,407083,99.85
ChalReplayResult,404286,99.17
RecFumbPlayer,403315,98.93
RecFumbTeam,403315,98.93
Interceptor,403168,98.89
FieldGoalDistance,398740,97.81
FieldGoalResult,398629,97.78
ExPointResult,397578,97.52


In [29]:
# Faixas percentuais usadas na aula
valores_ausentes_50   = ausentes_pct[ausentes_pct >= 50].index.tolist()
valores_ausentes_30_50 = ausentes_pct[(ausentes_pct >= 30) & (ausentes_pct < 50)].index.tolist()
valores_ausentes_20_30 = ausentes_pct[(ausentes_pct >= 20) & (ausentes_pct < 30)].index.tolist()
valores_ausentes_10_20 = ausentes_pct[(ausentes_pct >= 10) & (ausentes_pct < 20)].index.tolist()
valores_ausentes_5_10  = ausentes_pct[(ausentes_pct >= 5) & (ausentes_pct < 10)].index.tolist()
valores_ausentes_lt5   = ausentes_pct[(ausentes_pct > 0) & (ausentes_pct < 5)].index.tolist()

print(f">= 50%        : {len(valores_ausentes_50)} colunas")
print(f"30% a < 50%   : {len(valores_ausentes_30_50)} colunas")
print(f"20% a < 30%   : {len(valores_ausentes_20_30)} colunas")
print(f"10% a < 20%   : {len(valores_ausentes_10_20)} colunas")
print(f"5% a < 10%    : {len(valores_ausentes_5_10)} colunas")
print(f"> 0% e < 5%   : {len(valores_ausentes_lt5)} colunas")

>= 50%        : 33 colunas
30% a < 50%   : 1 colunas
20% a < 30%   : 0 colunas
10% a < 20%   : 1 colunas
5% a < 10%    : 12 colunas
> 0% e < 5%   : 18 colunas
